In [4]:
import base64
import hashlib
import secrets
from urllib.parse import urlencode

import httpx

BASE_URL = "http://localhost:8081"
REALM = "libris"
CLIENT_ID = "libris"
REDIRECT_URI = "http://localhost:8081/callback"

AUTH_URL = f"{BASE_URL}/realms/{REALM}/protocol/openid-connect/auth"
TOKEN_URL = f"{BASE_URL}/realms/{REALM}/protocol/openid-connect/token"


def iniciar_login() -> tuple[str, str, str]:
    verifier = secrets.token_urlsafe(64)
    challenge = (
        base64.urlsafe_b64encode(hashlib.sha256(verifier.encode()).digest())
        .rstrip(b"=")
        .decode()
    )
    state = secrets.token_urlsafe(32)

    params = {
        "client_id": CLIENT_ID,
        "response_type": "code",
        "scope": "openid profile email",
        "redirect_uri": REDIRECT_URI,
        "state": state,
        "code_challenge": challenge,
        "code_challenge_method": "S256",
    }
    return f"{AUTH_URL}?{urlencode(params)}", verifier, state


async def trocar_code_por_tokens(code: str, verifier: str) -> dict:
    async with httpx.AsyncClient(timeout=10) as client:
        response = await client.post(
            TOKEN_URL,
            data={
                "grant_type": "authorization_code",
                "client_id": CLIENT_ID,
                "code": code,
                "redirect_uri": REDIRECT_URI,
                "code_verifier": verifier,
            },
        )
        response.raise_for_status()
        return response.json()

In [5]:
import httpx

BASE_URL = "http://localhost:8081"
REALM = "libris"
CLIENT_ID = "libris"

TOKEN_URL = (
    f"{BASE_URL}/realms/{REALM}/protocol/openid-connect/token"
)


async def obter_token(username: str, password: str) -> dict:
    async with httpx.AsyncClient(timeout=10) as client:
        response = await client.post(
            TOKEN_URL,
            data={
                "grant_type": "password",
                "client_id": CLIENT_ID,
                "username": username,
                "password": password,
                "scope": "openid profile email",
            },
        )
        response.raise_for_status()
        return response.json()

In [6]:
tokens = await obter_token("libris-admin", "8486")
access_token = tokens["access_token"]

HTTPStatusError: Client error '401 Unauthorized' for url 'http://localhost:8081/realms/libris/protocol/openid-connect/token'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

In [1]:
import base64
import getpass
import hashlib
import secrets
import webbrowser
from urllib.parse import parse_qs, urlencode, urlsplit

import httpx

ISSUER = "http://keycloak.localhost:8081/realms/libris"
CALLBACK = "http://127.0.0.1:8765/callback"
CLIENT_ID = "libris-web"

verifier = secrets.token_urlsafe(64)
challenge = base64.urlsafe_b64encode(
    hashlib.sha256(verifier.encode("ascii")).digest()
).rstrip(b"=").decode("ascii")
state = secrets.token_urlsafe(32)

authorization_url = (
    f"{ISSUER}/protocol/openid-connect/auth?"
    + urlencode({
        "client_id": CLIENT_ID,
        "redirect_uri": CALLBACK,
        "response_type": "code",
        "scope": "openid profile",
        "state": state,
        "code_challenge": challenge,
        "code_challenge_method": "S256",
    })
)

print("Faça login no navegador que será aberto.")
webbrowser.open(authorization_url)

# Não há servidor nessa porta. Após o login, o navegador mostrará
# uma falha de conexão. Copie a URL completa da barra de endereços.
redirect = getpass.getpass(
    "Cole a URL completa após o redirecionamento (entrada oculta): "
).strip()

parsed = urlsplit(redirect)
if f"{parsed.scheme}://{parsed.netloc}{parsed.path}" != CALLBACK:
    raise SystemExit("Callback inesperado.")

query = parse_qs(parsed.query)
states = query.get("state", [])
codes = query.get("code", [])

if len(states) != 1 or not secrets.compare_digest(states[0], state):
    raise SystemExit("State inválido.")
if "error" in query or len(codes) != 1:
    raise SystemExit("Login não retornou um código válido.")

with httpx.Client(timeout=15, trust_env=False) as client:
    response = client.post(
        f"{ISSUER}/protocol/openid-connect/token",
        data={
            "grant_type": "authorization_code",
            "client_id": CLIENT_ID,
            "redirect_uri": CALLBACK,
            "code": codes[0],
            "code_verifier": verifier,
        },
    )
    if response.status_code != 200:
        raise SystemExit(f"Troca do código falhou: HTTP {response.status_code}")

    token = response.json()["access_token"]

    me = client.get(
        "http://localhost:8000/api/v1/identity/me",
        headers={"Authorization": f"Bearer {token}"},
    )
    print("GET /identity/me:", me.status_code)
    if me.is_success:
        print(me.json())

    input("Pressione Enter para exibir o token e copiá-lo para o Swagger.")
    print(token)

Faça login no navegador que será aberto.


SystemExit: State inválido.

/home/inacio/dev/libris/apps/api/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
